# Oracle Layout Injection — Rotation Questions Only

**Hypothesis:** The rotation bottleneck is a *reasoning* failure, not a representation failure.
LLaVA's patch features already encode depth (80.4% probe accuracy), but the model cannot
compose that information into a rotated reference frame when answering questions.

**Test:** For each of the 200 rotation questions, prepend an explicit per-image depth-zone
description derived from DepthAnything v2 into the text prompt. This hands the model
the geometric layout as text, bypassing the need to infer it from images.

If rotation accuracy improves substantially (e.g., >5pp over baseline 34.5%), it confirms
the model *can* reason about spatial transformations when layout is made explicit —
the bottleneck is in the visual-to-symbolic translation, not in spatial reasoning itself.

**Depth description format:** Each image is divided into a 3×3 grid of zones.
Zones are ranked from nearest to farthest (lower depth value = closer).
Example: `"Image 1 depth (near→far): center, bottom-left, top-right, ..."`

**Runtime:** A100 GPU. Only 200 samples (rotation setting) — ~10 min.

Output: `/content/oracle_layout_results.jsonl`

In [1]:
# ── 1. Install ───────────────────────────────────────────────────────────────
!pip install -q transformers>=4.45.0 accelerate>=0.27.0 pillow tqdm

In [2]:
# ── 2. Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH  = "/content/drive/MyDrive/MindCube/data/raw/MindCube_tinybench.jsonl"
IMAGE_ROOT = "/content/drive/MyDrive/MindCube/data/"
LLAVA_ID   = "/content/drive/MyDrive/models/llava-onevision-qwen2-7b-ov-hf"
DA_PATH    = "/content/drive/MyDrive/models/depth-anything-v2-small-hf"

import pathlib
assert pathlib.Path(DATA_PATH).exists()
print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


In [3]:
# ── 3. Load models ───────────────────────────────────────────────────────────
import torch, gc, json, re
import numpy as np
from PIL import Image
from pathlib import Path
from transformers import (
    LlavaOnevisionForConditionalGeneration, AutoProcessor, pipeline
)
from tqdm.notebook import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# DepthAnything v2
print("Loading DepthAnything ...")
depth_pipe = pipeline(
    task="depth-estimation", model=DA_PATH,
    device=0 if device == "cuda" else -1,
)

# LLaVA-OneVision
print("Loading LLaVA ...")
processor = AutoProcessor.from_pretrained(LLAVA_ID)
model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    LLAVA_ID, torch_dtype=torch.float16, device_map="auto", attn_implementation="sdpa",
)
model.eval()
processor.image_processor.do_image_splitting = False

gc.collect()
torch.cuda.empty_cache()
print(f"Models loaded. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Device: cuda
Loading DepthAnything ...


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading LLaVA ...


The image processor of type `LlavaOnevisionImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/765 [00:00<?, ?it/s]

Models loaded. GPU: 16.2 GB


In [4]:
# ── 4. Depth-to-text helper ───────────────────────────────────────────────────
# Zone names for a 3×3 grid (row-major, top-left origin)
_ZONE_NAMES = [
    "top-left",    "top-center",    "top-right",
    "middle-left", "center",        "middle-right",
    "bottom-left", "bottom-center", "bottom-right",
]


def depth_to_text(pil_image: Image.Image, image_label: str) -> str:
    """
    Run DepthAnything on an image, divide into 3x3 zones,
    rank zones from nearest (smallest depth value) to farthest,
    and return a one-line text description.

    Example output:
      "Image 2 depth (nearest to farthest): center, bottom-left, top-right, ..."
    """
    out = depth_pipe(pil_image)
    depth_arr = np.array(out["depth"], dtype=np.float32)  # (H, W)

    H, W = depth_arr.shape
    h3, w3 = H // 3, W // 3

    zone_means = []
    for r in range(3):
        for c in range(3):
            zone = depth_arr[r*h3:(r+1)*h3, c*w3:(c+1)*w3]
            zone_means.append(float(zone.mean()))

    # Sort zones: smaller depth value = nearer (DepthAnything: smaller = closer)
    ranked = sorted(range(9), key=lambda i: zone_means[i])
    zone_str = ", ".join(_ZONE_NAMES[i] for i in ranked)
    return f"{image_label} depth (nearest→farthest): {zone_str}"


# Sanity check
_img = Image.new("RGB", (300, 300), color=(100, 150, 200))
print(depth_to_text(_img, "Image 1"))

Image 1 depth (nearest→farthest): center, middle-right, middle-left, top-center, bottom-center, top-left, top-right, bottom-right, bottom-left


In [5]:
# ── 5. Answer extraction + inference helpers ──────────────────────────────────
_TAG  = re.compile(r"<answer>\s*([A-E])", re.I)
_DECL = re.compile(r"(?:the\s+answer\s+is|answer\s*:)\s*([A-E])\.?", re.I)
_LINE = re.compile(r"^\s*([A-E])[\.):]?\s*$", re.I | re.M)
_ANY  = re.compile(r"([A-E])", re.I)

def extract_answer(text):
    for pat in [_TAG, _DECL]:
        m = pat.search(text)
        if m:
            return m.group(1).upper()
    for pat in [_LINE, _ANY]:
        ms = pat.findall(text)
        if ms:
            return ms[-1].upper()
    return None


_HEADER = "Look at these images carefully. They show a scene from different viewpoints.\n\n"
_FOOTER = "\n\nAnswer with one letter only (A, B, C, or D)."


@torch.inference_mode()
def generate(images, prompt, max_new_tokens=128):
    content = [{"type": "image"} for _ in images] + [{"type": "text", "text": prompt}]
    conversation = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(images=images, text=text, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True).strip()


print("Helpers ready.")

Helpers ready.


In [6]:
# ── 6. Load rotation questions only ──────────────────────────────────────────
rotation_samples = []
with open(DATA_PATH) as f:
    for line in f:
        rec = json.loads(line)
        if rec["id"].startswith("rotation"):
            rotation_samples.append(rec)

print(f"Rotation questions: {len(rotation_samples)}  (baseline acc: 34.5%)")

Rotation questions: 200  (baseline acc: 34.5%)


In [7]:
# ── 7. Sanity check (3 samples) ──────────────────────────────────────────────
for rec in rotation_samples[:3]:
    images = [Image.open(Path(IMAGE_ROOT) / r).convert("RGB") for r in rec["images"]]
    depth_lines = [depth_to_text(img, f"Image {i+1}") for i, img in enumerate(images)]
    depth_block = "\n".join(depth_lines)
    prompt = _HEADER + depth_block + "\n\n" + rec["question"] + _FOOTER
    raw = generate(images, prompt)
    pred = extract_answer(raw)
    print(f"[{rec['id']}]  gt={rec['gt_answer']}  pred={pred}  raw={repr(raw[:60])}")
    for img in images:
        img.close()

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[rotation_group000_q3_6]  gt=A  pred=B  raw='B'


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[rotation_group015_q1_1]  gt=B  pred=B  raw='B'


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


[rotation_group005_q1_2]  gt=A  pred=B  raw='B'


In [8]:
# ── 8. Full evaluation on all 200 rotation questions ─────────────────────────
OUT_PATH = "/content/oracle_layout_results.jsonl"
BASELINE_ROTATION = 0.345

results = []
with open(OUT_PATH, "w") as f_out:
    for rec in tqdm(rotation_samples):
        images = []
        try:
            images = [Image.open(Path(IMAGE_ROOT) / r).convert("RGB") for r in rec["images"]]
            depth_lines = [depth_to_text(img, f"Image {i+1}") for i, img in enumerate(images)]
            depth_block = "\n".join(depth_lines)
            prompt = _HEADER + depth_block + "\n\n" + rec["question"] + _FOOTER
            raw = generate(images, prompt)
            predicted = extract_answer(raw)
            error = None
        except Exception as e:
            raw, predicted, error = "", None, str(e)
            tqdm.write(f"[WARN] {rec['id']}: {str(e)[:100]}")

        for img in images:
            img.close()

        gt = (rec["gt_answer"] or "").upper()
        result = {
            "id": rec["id"],
            "gt_answer": gt,
            "predicted": predicted,
            "correct": predicted is not None and predicted == gt,
            "raw_output": raw,
            **(({"error": error}) if error else {}),
        }
        results.append(result)
        f_out.write(json.dumps(result) + "\n")

print(f"Done. Saved to {OUT_PATH}")

  0%|          | 0/200 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_to

Done. Saved to /content/oracle_layout_results.jsonl


In [9]:
# ── 9. Metrics ────────────────────────────────────────────────────────────────
correct = sum(1 for r in results if r["correct"])
total   = len(results)
acc     = correct / total if total else 0.0
delta   = acc - BASELINE_ROTATION
unanswered = sum(1 for r in results if r["predicted"] is None)

print(f"\n{'='*48}")
print(f"  Oracle Layout Injection — Rotation only")
print(f"{'='*48}")
print(f"  Baseline accuracy  : {BASELINE_ROTATION:.3f}  (69/200)")
print(f"  Oracle accuracy    : {acc:.3f}  ({correct}/{total})")
print(f"  Delta              : {delta:+.3f}")
print(f"  Unanswered         : {unanswered}/{total}")
print(f"{'='*48}")
print()
if delta > 0.05:
    print(">> Substantial improvement: explicit layout helps reasoning — REASONING BOTTLENECK confirmed.")
elif delta > 0.02:
    print(">> Modest improvement: some benefit from explicit layout.")
else:
    print(">> Negligible improvement: explicit layout does not help reasoning.")
    print("   Possible explanations: model cannot parse text-format depth cues, or")
    print("   depth zone ordering is too coarse to be useful for these questions.")


  Oracle Layout Injection — Rotation only
  Baseline accuracy  : 0.345  (69/200)
  Oracle accuracy    : 0.335  (67/200)
  Delta              : -0.010
  Unanswered         : 0/200

>> Negligible improvement: explicit layout does not help reasoning.
   Possible explanations: model cannot parse text-format depth cues, or
   depth zone ordering is too coarse to be useful for these questions.


In [10]:
# ── 10. Download results ──────────────────────────────────────────────────────
from google.colab import files
files.download(OUT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>